In [1]:
import sys
sys.executable

'/Users/mariabelengatti/calima-canarias/spikes/.venv/bin/python'

In [2]:
import pandas as pd

url = "https://ereporting.blob.core.windows.net/downloadservice/metadata.csv"
meta = pd.read_csv(url)
meta.shape

ParserError: Error tokenizing data. C error: Expected 1 fields in line 49, saw 2


In [3]:
import requests

r = requests.get(url, stream=True)
for i, line in enumerate(r.iter_lines()):
    print(line[:200])
    if i >= 2:
        break

b'Countrycode\tTimezone\tNamespace\tAirQualityNetwork\tAirQualityStation\tAirQualityStationEoICode\tAirQualityStationNatCode\tSamplingPoint\tSamplingProcess\tSample\tAirPollutantCode\tAirPollutant\tProjection\tLongi'
b'AL\thttp://dd.eionet.europa.eu/vocabulary/aq/timezone/UTC+01\tAL.NEA.AQ\tNET-AL002A\tSTA-AL0201A\tAL0201A\tAL360A01\tSPO-AL0201A_00001_100\tSPP-AL_A_UV-FL_API100E\tSPO_F-AL0201A_00001_100_100\thttp://dd.eionet.'
b'AL\thttp://dd.eionet.europa.eu/vocabulary/aq/timezone/UTC+01\tAL.NEA.AQ\tNET-AL002A\tSTA-AL0201A\tAL0201A\tAL360A01\tSPO-AL0201A_00005_100\tSPP-AL_A_light-scat_GRIMM-EDM180\tSPO_F-AL0201A_00005_100_100\thttp://'


In [4]:
meta = pd.read_csv(url, sep="\t")
meta.shape

/var/folders/0x/t6d1bysn3cbdrdy40n0tlr400000gn/T/ipykernel_23077/2680034940.py:1: DtypeWarning: Columns (20: EquivalenceDemonstrated) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(url, sep="\t")


(44523, 25)

In [5]:
meta = pd.read_csv(url, sep="\t", low_memory=False)

In [6]:
meta.columns.tolist()

['Countrycode',
 'Timezone',
 'Namespace',
 'AirQualityNetwork',
 'AirQualityStation',
 'AirQualityStationEoICode',
 'AirQualityStationNatCode',
 'SamplingPoint',
 'SamplingProcess',
 'Sample',
 'AirPollutantCode',
 'AirPollutant',
 'Projection',
 'Longitude',
 'Latitude',
 'Altitude',
 'MeasurementType',
 'MeasurementMethod',
 'AirQualityStationType',
 'AirQualityStationArea',
 'EquivalenceDemonstrated',
 'MeasurementEquipment',
 'InletHeight',
 'BuildingDistance',
 'KerbDistance']

In [7]:
import pandas as pd

# 1. Cargar la tabla de metadatos (todos los puntos de medición de Europa)
url = "https://ereporting.blob.core.windows.net/downloadservice/metadata.csv"
meta = pd.read_csv(url, sep="\t", low_memory=False)

# 2. El recuadro de Canarias, con un poco de margen
#    Sur: El Hierro (~27.6)  ·  Norte: islotes de Lanzarote (~29.4)
#    Oeste: El Hierro (~ -18.2) · Este: Lanzarote (~ -13.3)
LAT_MIN, LAT_MAX = 27.5, 29.5
LON_MIN, LON_MAX = -18.5, -13.0

# 3. Filtrar: España + PM10 + dentro del recuadro
canarias_pm10 = meta[
    (meta["Countrycode"] == "ES")
    & (meta["AirPollutant"] == "PM10")
    & (meta["Latitude"].between(LAT_MIN, LAT_MAX))
    & (meta["Longitude"].between(LON_MIN, LON_MAX))
]

# 4. Mostrar resultados
print(canarias_pm10.shape)
print(canarias_pm10["Timezone"].unique())
print(canarias_pm10["AirQualityNetwork"].unique())

(0, 25)
<ArrowStringArray>
[]
Length: 0, dtype: str
<ArrowStringArray>
[]
Length: 0, dtype: str


In [8]:
meta["AirPollutant"].unique()[:30]

<ArrowStringArray>
[                   'SO2',                   'PM10',                     'O3',
                    'NO2',                     'CO',                   'C6H6',
                     'NO',                  'PM2.5',                    'SPM',
                     'Pb',                     'Cd',                     'Ni',
                     'As',               'C6H5-CH3',  '(CH3)2-CH-CH2-CH2-CH3',
              'C6H5-C2H5',                  'C7H16',                  'C6H14',
 '(CH3)3-C-CH2-CH-(CH3)2',       'H3C-CH2-CH(CH3)2',         'm,p-C6H4(CH3)2',
                  'C8H18',          'o-C6H4-(CH3)2',         'H3C-(CH2)3-CH3',
                    'PM1',     'H2C=CH-CH2-CH2-CH3',      'H3C-HC=CH-CH2-CH3',
       '1,2,4-C6H3(CH3)3',       '1,2,3-C6H3(CH3)3',       '1,3,5-C6H3(CH3)3']
Length: 30, dtype: str

In [9]:
print("ES:        ", (meta["Countrycode"] == "ES").sum())
print("PM10:      ", (meta["AirPollutant"] == "PM10").sum())
print("Latitud:   ", meta["Latitude"].between(LAT_MIN, LAT_MAX).sum())
print("Longitud:  ", meta["Longitude"].between(LON_MIN, LON_MAX).sum())
print("Tipos:     ", meta["Latitude"].dtype, meta["Longitude"].dtype)

ES:         7011
PM10:       5242
Latitud:    0
Longitud:   0
Tipos:      float64 float64


In [10]:
meta[["Latitude", "Longitude"]].describe()

,Latitude,Longitude
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


In [11]:
es = meta[meta["Countrycode"] == "ES"]
es.sort_values("Latitude")[
    ["AirQualityStation", "AirQualityNetwork", "Latitude", "Longitude", "Projection"]
].head(10)

,AirQualityStation,AirQualityNetwork,Latitude,Longitude,Projection
14147,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14148,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14149,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14150,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14151,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14152,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14153,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14154,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14155,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979
14156,STA-ES0001R,NET-ES058A,NaN,NaN,EPSG:4979


In [12]:
r = requests.get(url, stream=True)
lines = r.iter_lines()

header = next(lines).decode().split("\t")   # primera línea: nombres de columnas
row = next(lines).decode().split("\t")      # segunda línea: primer dato

for name, value in zip(header, row):
    print(f"{name:28} | {value}")

Countrycode                  | AL
Timezone                     | http://dd.eionet.europa.eu/vocabulary/aq/timezone/UTC+01
Namespace                    | AL.NEA.AQ
AirQualityNetwork            | NET-AL002A
AirQualityStation            | STA-AL0201A
AirQualityStationEoICode     | AL0201A
AirQualityStationNatCode     | AL360A01
SamplingPoint                | SPO-AL0201A_00001_100
SamplingProcess              | SPP-AL_A_UV-FL_API100E
Sample                       | SPO_F-AL0201A_00001_100_100
AirPollutantCode             | http://dd.eionet.europa.eu/vocabulary/aq/pollutant/1
AirPollutant                 | SO2
Projection                   | EPSG:4979
Longitude                    | nan
Latitude                     | nan
Altitude                     | 162
MeasurementType              | automatic
MeasurementMethod            | UV-FL
AirQualityStationType        | traffic
AirQualityStationArea        | urban
EquivalenceDemonstrated      | 
MeasurementEquipment         | 
InletHeight             

In [13]:
EEA metadata.csv (nov 2026): Latitude y Longitude vacías en todas las filas, aunque Projection dice EPSG:4979. Las coordenadas hay que sacarlas de otra fuente.

SyntaxError: invalid syntax (1327674636.py, line 1)

EEA metadata.csv (nov 2026): Latitude y Longitude vacías en todas las filas, aunque Projection dice EPSG:4979. Las coordenadas hay que sacarlas de otra fuente.

In [14]:
es["Timezone"].value_counts()

Timezone
http://dd.eionet.europa.eu/vocabulary/aq/timezone/UTC+01    7011
Name: count, dtype: int64

In [15]:
es["AirQualityStationNatCode"].head(10)

14147    45153999
14148    45153999
14149    45153999
14150    45153999
14151    45153999
14152    45153999
14153    45153999
14154    45153999
14155    45153999
14156    45153999
Name: AirQualityStationNatCode, dtype: str

In [16]:
es_pm10 = es[es["AirPollutant"] == "PM10"]
canarias = es_pm10[es_pm10["AirQualityStationNatCode"].str.startswith(("35", "38"))]

print("Filas PM10 en Canarias:", len(canarias))
print("Estaciones distintas:  ", canarias["AirQualityStation"].nunique())
print("Redes:                 ", canarias["AirQualityNetwork"].unique())
print("Altitud vacía en:      ", canarias["Altitude"].isna().sum(), "de", len(canarias))

Filas PM10 en Canarias: 60
Estaciones distintas:   52
Redes:                  <ArrowStringArray>
['NET-ES205A']
Length: 1, dtype: str
Altitud vacía en:       0 de 60


In [17]:
canarias[
    ["AirQualityStation", "AirQualityStationNatCode", "AirQualityStationType",
     "AirQualityStationArea", "Altitude"]
].drop_duplicates("AirQualityStation")

,AirQualityStation,AirQualityStationNatCode,AirQualityStationType,AirQualityStationArea,Altitude
14765,STA-ES0197A,35026002,industrial,urban,56
15310,STA-ES0873A,35016010,industrial,suburban,22
15337,STA-ES0886A,35026001,industrial,urban,52
15877,STA-ES1131A,38038010,industrial,urban,75
15890,STA-ES1132A,38038011,industrial,urban,95
15900,STA-ES1133A,38011003,industrial,suburban,145
17401,STA-ES1451A,35016011,industrial,urban,5
17986,STA-ES1573A,35016012,traffic,urban,9
19170,STA-ES1741A,35019002,traffic,urban,17
19177,STA-ES1742A,35019001,traffic,urban,18


In [18]:
estaciones = canarias.drop.duplicates("AirQualityStation')

SyntaxError: unterminated string literal (detected at line 1) (3331538779.py, line 1)

In [19]:
estaciones = canarias.drop_duplicates("AirQualityStation")
estaciones["AirQualityStationType"].value_counts(dropna=False)

AirQualityStationType
industrial    26
background    16
traffic        6
NaN            4
Name: count, dtype: int64

In [20]:
canarias.to_csv("../data/raw/canarias_pm10_samplingpoints.csv", index=False)

OSError: Cannot save file into a non-existent directory: '../data/raw'

In [21]:
from pathlib import Path

Path("../data/raw").mkdir(parents=True, exist_ok=True)
canarias.to_csv("../data/raw/canarias_pm10_samplingpoints.csv", index=False)